In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA gold;

In [0]:
CREATE TABLE IF NOT EXISTS characters_behavior_daily (
  -- Snapshot metadata
  snapshot_date         DATE,

  -- Character identity
  character_id          STRING,
  world_id              STRING,

  -- Character attributes
  account_status        STRING,
  sex                   STRING,
  vocation              STRING,

  -- Progression
  level                 INT,
  level_delta_1d        INT,
  level_delta_7d        INT,
  level_delta_30d       INT,
  level_delta_90d       INT,

  -- Activity
  first_seen_date       DATE,
  last_login_date       DATE,
  days_since_last_login INT,
  is_active_1d          BOOLEAN,
  is_active_7d          BOOLEAN,
  is_active_30d         BOOLEAN,
  is_active_90d         BOOLEAN,

  -- Lifecycle
  lifecycle_stage       STRING,
  lifecycle_event       STRING,
  
  -- Social and account signals
  in_guild              BOOLEAN,
  owns_house            BOOLEAN,
  is_married            BOOLEAN,
  loyalty_title         STRING,
  achievement_points    INT,

  -- Processing metadata
  processed_at          TIMESTAMP
)
USING DELTA
PARTITIONED BY (snapshot_date);

In [0]:
COMMENT ON TABLE  characters_behavior_daily                       IS 'Daily behavioral snapshot of Tibia characters used for longitudinal community health analysis.';
COMMENT ON COLUMN characters_behavior_daily.snapshot_date         IS 'Reference date of the data snapshot.';
COMMENT ON COLUMN characters_behavior_daily.character_id          IS 'Stable character identifier generated by the identity resolution process. Used to track the same character across historical snapshots.';
COMMENT ON COLUMN characters_behavior_daily.world_id              IS 'SHA1 identifier generated from the normalized world name. Used as the stable world identifier.';
COMMENT ON COLUMN characters_behavior_daily.account_status        IS 'Indicates whether the character account is Free or Premium.';
COMMENT ON COLUMN characters_behavior_daily.sex                   IS 'Current sex of the character on the snapshot date.';
COMMENT ON COLUMN characters_behavior_daily.vocation              IS 'Current vocation of the character on the snapshot date.';
COMMENT ON COLUMN characters_behavior_daily.level                 IS 'Current level of the character on the snapshot date.';
COMMENT ON COLUMN characters_behavior_daily.level_delta_1d        IS 'Level change compared to the previous day.';
COMMENT ON COLUMN characters_behavior_daily.level_delta_7d        IS 'Level change compared to 7 days ago.';
COMMENT ON COLUMN characters_behavior_daily.level_delta_30d       IS 'Level change compared to 30 days ago.';
COMMENT ON COLUMN characters_behavior_daily.level_delta_90d       IS 'Level change compared to 90 days ago.';
COMMENT ON COLUMN characters_behavior_daily.first_seen_date       IS 'Earliest date on which the character was observed in the dataset.';
COMMENT ON COLUMN characters_behavior_daily.last_login_date       IS 'Most recent known login date of the character.';
COMMENT ON COLUMN characters_behavior_daily.days_since_last_login IS 'Number of days since the character\'s last known login.';
COMMENT ON COLUMN characters_behavior_daily.is_active_1d          IS 'Indicates whether the character was active within the last 1 day.';
COMMENT ON COLUMN characters_behavior_daily.is_active_7d          IS 'Indicates whether the character was active within the last 7 days.';
COMMENT ON COLUMN characters_behavior_daily.is_active_30d         IS 'Indicates whether the character was active within the last 30 days.';
COMMENT ON COLUMN characters_behavior_daily.is_active_90d         IS 'Indicates whether the character was active within the last 90 days.';
COMMENT ON COLUMN characters_behavior_daily.lifecycle_stage       IS 'Character lifecycle classification on the snapshot date. Possible values: new, returning, active, dormant, churned.';
COMMENT ON COLUMN characters_behavior_daily.lifecycle_event       IS 'Lifecycle transition event detected on the snapshot date. Possible values: new, returning, dormant, churned, or NULL when no transition occurs.';
COMMENT ON COLUMN characters_behavior_daily.in_guild              IS 'Indicates whether the character is currently a member of a guild.';
COMMENT ON COLUMN characters_behavior_daily.owns_house            IS 'Indicates whether the character owns at least one house at the time of the snapshot.';
COMMENT ON COLUMN characters_behavior_daily.is_married            IS 'Indicates whether the character is currently married.';
COMMENT ON COLUMN characters_behavior_daily.loyalty_title         IS 'The loyalty title associated with the character account.';
COMMENT ON COLUMN characters_behavior_daily.achievement_points    IS 'Total achievement points accumulated by the character.';
COMMENT ON COLUMN characters_behavior_daily.processed_at          IS 'Timestamp when this record was processed into the Gold layer (UTC).';

In [0]:
CREATE TABLE IF NOT EXISTS characters_behavior_periodic (
  -- Aggregation period
  granularity           STRING,
  period_start          DATE,
  period_end            DATE,
  period_days           INT,
  observed_days         INT,
  is_partial_period     BOOLEAN,
  period_status         STRING,

  -- Character identity
  character_id          STRING,
  world_id              STRING,

  -- Character attributes
  account_status        STRING,
  sex                   STRING,
  vocation              STRING,

  -- Progression
  level_start           INT,
  level                 INT,
  level_delta_period    INT,
  level_delta_short     INT,
  level_delta_medium    INT,
  level_delta_long      INT,
  level_delta_xlong     INT,

  -- Activity
  first_seen_date       DATE,
  last_login_date       DATE,
  days_since_last_login INT,
  is_active_in_period   BOOLEAN,
  is_active_1d          BOOLEAN,
  is_active_7d          BOOLEAN,
  is_active_30d         BOOLEAN,
  is_active_90d         BOOLEAN,

  -- Lifecycle
  lifecycle_stage       STRING,
  lifecycle_event       STRING,
  new_events            INT,
  returning_events      INT,
  dormant_events        INT,
  churned_events        INT,

  -- Social and account signals
  in_guild              BOOLEAN,
  owns_house            BOOLEAN,
  is_married            BOOLEAN,
  loyalty_title         STRING,
  achievement_points    INT,

  -- Processing metadata
  processed_at          TIMESTAMP
);

In [0]:
COMMENT ON TABLE  characters_behavior_periodic                       IS 'Periodic behavioral snapshot of Tibia characters aggregated into standardized time windows for longitudinal community health, engagement, retention, and lifecycle analysis.';
COMMENT ON COLUMN characters_behavior_periodic.granularity           IS 'Aggregation level of the behavioral period. Supported values: Day, Week, Month, Quarter.';
COMMENT ON COLUMN characters_behavior_periodic.period_start          IS 'Start date of the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.period_end            IS 'End date of the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.period_days           IS 'Expected number of calendar days in the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.observed_days         IS 'Number of snapshot dates observed in the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.is_partial_period     IS 'Indicates whether the aggregation period contains incomplete snapshot coverage.';
COMMENT ON COLUMN characters_behavior_periodic.period_status         IS 'Coverage classification of the aggregation period. Supported values: complete, partial_start, partial_current, partial_missing.';
COMMENT ON COLUMN characters_behavior_periodic.character_id          IS 'Stable character identifier generated by the identity resolution process. Used to track the same character across historical snapshots.';
COMMENT ON COLUMN characters_behavior_periodic.world_id              IS 'SHA1 identifier generated from the normalized world name. Used as the stable world identifier.';
COMMENT ON COLUMN characters_behavior_periodic.account_status        IS 'Indicates whether the character account is Free or Premium.';
COMMENT ON COLUMN characters_behavior_periodic.sex                   IS 'The character\'s sex during the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.vocation              IS 'The character\'s vocation during the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.level_start           IS 'Character level observed at the beginning of the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.level                 IS 'Character level observed at the end of the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.level_delta_period    IS 'Level change during the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.level_delta_short     IS 'Level change compared to the previous equivalent short-term comparison window.';
COMMENT ON COLUMN characters_behavior_periodic.level_delta_medium    IS 'Level change compared to the previous equivalent medium-term comparison window.';
COMMENT ON COLUMN characters_behavior_periodic.level_delta_long      IS 'Level change compared to the previous equivalent long-term comparison window.';
COMMENT ON COLUMN characters_behavior_periodic.level_delta_xlong     IS 'Level change compared to the previous equivalent extra-long-term comparison window.';
COMMENT ON COLUMN characters_behavior_periodic.first_seen_date       IS 'Earliest date on which the character was observed in the dataset.';
COMMENT ON COLUMN characters_behavior_periodic.last_login_date       IS 'Most recent known login date of the character.';
COMMENT ON COLUMN characters_behavior_periodic.days_since_last_login IS 'Number of days since the character\'s last known login.';
COMMENT ON COLUMN characters_behavior_periodic.is_active_in_period   IS 'Indicates whether the character logged in at least once during the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.is_active_1d          IS 'Indicates whether the character was active within the last 1 day.';
COMMENT ON COLUMN characters_behavior_periodic.is_active_7d          IS 'Indicates whether the character was active within the last 7 days.';
COMMENT ON COLUMN characters_behavior_periodic.is_active_30d         IS 'Indicates whether the character was active within the last 30 days.';
COMMENT ON COLUMN characters_behavior_periodic.is_active_90d         IS 'Indicates whether the character was active within the last 90 days.';
COMMENT ON COLUMN characters_behavior_periodic.lifecycle_stage       IS 'Character lifecycle classification at the end of the aggregation period. Possible values: new, returning, active, dormant, churned.';
COMMENT ON COLUMN characters_behavior_periodic.lifecycle_event       IS 'Highest-priority lifecycle transition event observed during the aggregation period. Possible values: new, returning, dormant, churned, or NULL when no transition occurs.';
COMMENT ON COLUMN characters_behavior_periodic.new_events            IS 'Number of new lifecycle transition events observed during the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.returning_events      IS 'Number of returning lifecycle transition events observed during the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.dormant_events        IS 'Number of dormant lifecycle transition events observed during the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.churned_events        IS 'Number of churned lifecycle transition events observed during the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.in_guild              IS 'Indicates whether the character was a member of a guild in the latest snapshot of the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.owns_house            IS 'Indicates whether the character owned at least one house in the latest snapshot of the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.is_married            IS 'Indicates whether the character was married in the latest snapshot of the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.loyalty_title         IS 'The loyalty title associated with the character account in the latest snapshot of the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.achievement_points    IS 'Total achievement points accumulated by the character in the latest snapshot of the aggregation period.';
COMMENT ON COLUMN characters_behavior_periodic.processed_at          IS 'Timestamp when this record was processed into the Gold layer (UTC).';